# VASI Spatial Audio Test

This notebook performs a **VASI (Virtual Auditory Space Identification) test** by playing a sound stimulus spatialised from 8 directions in order:

| Step | Direction | Azimuth (pyfar convention) |
|------|-----------|----------------------------|
| 1 | 0° (Front) | 0° |
| 2 | 45° Right | −45° (315°) |
| 3 | 90° Right | −90° (270°) |
| 4 | 135° Right | −135° (225°) |
| 5 | 180° (Back) | 180° |
| 6 | 135° Left | 135° |
| 7 | 90° Left | 90° |
| 8 | 45° Left | 45° |

> **Convention:** pyfar uses azimuth measured counter-clockwise from front when viewed from above, so **left = positive azimuth** and **right = negative azimuth**.

All directions are on the horizontal plane (elevation = 0°).

In [1]:
# Imports
import pyfar as pf
import numpy as np
import sounddevice as sd
import scipy.io.wavfile
import time

In [15]:
# ── Paths ─────────────────────────────────────────────────────────────────────
# Update these paths to match your local dataset
hrtf_name = 'dataset/dataset/P0001/HRTF/HRTF/48kHz/P0001_FreeFieldCompMinPhase_NoITD_48kHz.sofa'
audio     = 'dataset/dataset/audio/sinewave.wav'
# ──────────────────────────────────────────────────────────────────────────────

In [16]:
# Load the HRTF database
print('Loading HRTF database...')
data_ir, source_coordinates, receiver_coordinates = pf.io.read_sofa(hrtf_name)
fs = data_ir.sampling_rate
print(f'  Sampling rate : {fs} Hz')
print(f'  Measurements  : {source_coordinates.csize} directions')

Loading HRTF database...
SOFA file contained custom entries
----------------------------------
GLOBAL_ReceiverDescription, GLOBAL_RoomDescription, GLOBAL_RoomLocation, GLOBAL_SourceDescription, GLOBAL_EmitterDescription, MeasurementSourceAudioChannel
  Sampling rate : 48000.0 Hz
  Measurements  : 793 directions


In [17]:
# Load and normalise the audio stimulus
sr, x = scipy.io.wavfile.read(audio)
x = x.astype(np.float32) / np.max(np.abs(x))

# Mix down to mono
if x.ndim == 1:
    x_mono = x
else:
    x_mono = np.mean(x, axis=1)

print(f'Audio loaded  : {len(x_mono)/sr:.2f} s  @{sr} Hz')

Audio loaded  : 2.59 s  @48000 Hz


In [18]:
def spatialise(x_mono, azimuth_deg, elevation_deg=0.0):
    """
    Convolve a mono signal with the nearest HRIR for a given direction.

    Parameters
    ----------
    x_mono       : 1-D numpy array  (mono audio, float32)
    azimuth_deg  : float  azimuth in degrees  (0=front, +90=left, -90=right)
    elevation_deg: float  elevation in degrees (0 = horizontal plane)

    Returns
    -------
    stereo_out   : (N, 2) numpy array ready for sd.play()
    """
    az_rad = np.deg2rad(azimuth_deg)
    el_rad = np.deg2rad(elevation_deg)

    target = pf.Coordinates.from_spherical_elevation(az_rad, el_rad, 1.0)
    idx, *_ = source_coordinates.find_nearest(target, k=1)

    h_left  = data_ir.time[idx[0], 0]
    h_right = data_ir.time[idx[0], 1]

    out_left  = np.convolve(x_mono, h_left)
    out_right = np.convolve(x_mono, h_right)

    # Normalise to avoid clipping
    peak = np.max(np.abs([out_left, out_right]))
    if peak > 0:
        out_left  = out_left  / peak * 0.9
        out_right = out_right / peak * 0.9

    return np.column_stack((out_left, out_right))

In [ ]:
# ── VASI test directions ───────────────────────────────────────────────────────
# Each tuple: (label, azimuth_in_degrees)
vasi_directions = [
    ('Front (0°)',          0),
    ('45° Right',         -45),
    ('90° Right',         -90),
    ('135° Right',       -135),
    ('Back (180°)',        180),
    ('135° Left',         135),
    ('90° Left',           90),
    ('45° Left',           45),
]

# Gap of silence between stimuli (seconds)
GAP_SECONDS = 1.5

print('VASI test directions:')
for label, az in vasi_directions:
    print(f'  {label:20s}  azimuth = {az:+4d}°')

VASI test directions:
  Front (0°)            azimuth =   +0°
  45° Right             azimuth =  -45°
  90° Right             azimuth =  -90°
  135° Right            azimuth = -135°
  Back (180°)           azimuth = +180°
  135° Left             azimuth = +135°
  90° Left              azimuth =  +90°
  45° Left              azimuth =  +45°


In [20]:
# ── Run the VASI test ──────────────────────────────────────────────────────────
print('='*55)
print('  Starting VASI spatial audio test')
print('  Make sure you are wearing headphones!')
print('='*55)
time.sleep(2)  # Give a moment to put on headphones

for step, (label, az_deg) in enumerate(vasi_directions, start=1):
    print(f'\n[{step}/{len(vasi_directions)}]  Playing: {label}  (azimuth = {az_deg:+d}°)')

    stereo = spatialise(x_mono, azimuth_deg=az_deg, elevation_deg=0.0)
    sd.play(stereo, fs)
    sd.wait()

    # Short silence gap before the next stimulus
    if step < len(vasi_directions):
        time.sleep(GAP_SECONDS)

print('\n' + '='*55)
print('  VASI test complete!')
print('='*55)

  Starting VASI spatial audio test
  Make sure you are wearing headphones!

[1/8]  Playing: Front (0°)  (azimuth = +0°)

[2/8]  Playing: 45° Right  (azimuth = -45°)

[3/8]  Playing: 90° Right  (azimuth = -90°)

[4/8]  Playing: 135° Right  (azimuth = -135°)

[5/8]  Playing: Back (180°)  (azimuth = +180°)

[6/8]  Playing: 135° Left  (azimuth = +135°)

[7/8]  Playing: 90° Left  (azimuth = +90°)

[8/8]  Playing: 45° Left  (azimuth = +45°)

  VASI test complete!


In [24]:
#play a single direction interactively ────────────────────────────
AZIMUTH_DEG   =  -15  # degrees  (0=front, +90=left, -90=right)
ELEVATION_DEG =   0   # degrees  (0 = horizontal plane)

print(f'Playing: azimuth={AZIMUTH_DEG:+d}°  elevation={ELEVATION_DEG:+d}°')
stereo = spatialise(x_mono, AZIMUTH_DEG, ELEVATION_DEG)
sd.play(stereo, fs)
sd.wait()
print('Done.')

Playing: azimuth=-15°  elevation=+0°
Done.
